In [10]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR

# ==============================================================================
# Cell 1 & 2: Load Data
# ==============================================================================
df = pd.read_csv("machine_rul_dataset.csv")
print(df)

# ==============================================================================
# Cell 3 to 8: Basic EDA & Identifiers (Task a)
# ==============================================================================
print(df.columns)
df.info()
print(df.describe())
print(df.head(8))
print(df.duplicated().sum())
print(df.isnull().sum())

# Drop non-predictive identifier column
if "Record_ID" in df.columns:
    df.drop(columns=["Record_ID"], inplace=True)

# Continuous target column
target = "Remaining_Useful_Life_Hours"

# ==============================================================================
# Cell 19 to 25: Feature-Target Split & Train-Test Split
# ==============================================================================
X = df.drop(columns=[target])
y = df[target]

print(X.shape)
print(y.shape)
print(X.head())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(y_train.shape)

# ==============================================================================
# Cell 26: Preprocessing Pipelines (SimpleImputer + Scaler / Encoder)
# ==============================================================================
number = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
cate = X_train.select_dtypes(include=["object", "str"]).columns.tolist()

# Sub-pipeline for numerical columns: Median Imputation + Scaling
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Sub-pipeline for categorical columns: Mode Imputation + One-Hot Encoding
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combined Preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipe, number),
    ("cat", cat_pipe, cate)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# ==============================================================================
# Cell 45 to 48: Task (b) - Model 1 (SVR Without PCA) Direct Tuning
# ==============================================================================
svr = SVR(kernel="rbf")

param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=1
)

grid.fit(X_train_encoded, y_train)

best_model1 = grid.best_estimator_
y_pred_m1 = best_model1.predict(X_test_encoded)

print("Best Parameters:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

best_idx = grid.best_index_
print(best_idx)

print("--- Model 1 (SVR Without PCA) Test Evaluation ---")
print("R2 Score :", r2_score(y_test, y_pred_m1))
print("MAE      :", mean_absolute_error(y_test, y_pred_m1))
print("MSE      :", mean_squared_error(y_test, y_pred_m1))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, y_pred_m1)))

result_df = pd.DataFrame(grid.cv_results_)
print(result_df)

# ==============================================================================
# Cell 50: Task (c) - PCA Variance Selection (>= 95%)
# ==============================================================================
pca_full = PCA()
pca_full.fit(X_train_encoded)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cum_var >= 0.95) + 1

print("Original shape:", X_train_encoded.shape[1])
print("Components for >= 95% variance:", n_components_95)

pca = PCA(
    n_components=n_components_95,
    random_state=42
)

X_train_pca = pca.fit_transform(X_train_encoded)
X_test_pca = pca.transform(X_test_encoded)

print(pca.explained_variance_ratio_)
print(pca.singular_values_)

# ==============================================================================
# Cell 54 to 57: Task (c) - Model 2 (SVR With PCA) Direct Tuning
# ==============================================================================
svr_pca = SVR(kernel="rbf")

param_grid_pca = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto']
}

grid_pca = GridSearchCV(
    estimator=svr_pca,
    param_grid=param_grid_pca,
    cv=5,
    n_jobs=1,
    scoring="neg_root_mean_squared_error"
)

grid_pca.fit(X_train_pca, y_train)

best_model2 = grid_pca.best_estimator_
print(grid_pca.best_estimator_)
print(grid_pca.cv_results_)
print(grid_pca.best_score_)

y_pred_pca = grid_pca.predict(X_test_pca)
print(y_pred_pca)

print("--- Model 2 (SVR With PCA) Test Evaluation ---")
print("R2 Score :", r2_score(y_test, y_pred_pca))
print("MAE      :", mean_absolute_error(y_test, y_pred_pca))
print("MSE      :", mean_squared_error(y_test, y_pred_pca))
print("RMSE     :", np.sqrt(mean_squared_error(y_test, y_pred_pca)))

# ==============================================================================
# Cell 58 to 59: Task (d) - 5-Fold Validation Table
# ==============================================================================
scores1 = -cross_val_score(best_model1, X_train_encoded, y_train, cv=5, scoring="neg_root_mean_squared_error")
scores2 = -cross_val_score(best_model2, X_train_pca, y_train, cv=5, scoring="neg_root_mean_squared_error")

print("NO PCA:", scores1)
print("PCA:", scores2)

table_d = pd.DataFrame({
    "Fold": [1, 2, 3, 4, 5],
    "RMSE Without PCA": scores1,
    "RMSE With PCA": scores2
})
table_d["RMSE Difference"] = table_d["RMSE With PCA"] - table_d["RMSE Without PCA"]

print("\n--- 5-Fold Validation Table ---")
print(table_d.to_string(index=False))

# ==============================================================================
# Cell 70: Task (e) - Statistical Significance Test (Paired t-Test)
# ==============================================================================
m1 = scores1
m2 = scores2

paired_stat, paired_p = ttest_rel(m2, m1)
print(f"Paired t-test: stat = {paired_stat:.4f}, p = {paired_p:.4f}")
if paired_p < 0.05:
    print("Paired t-test: Significant difference (p < 0.05)")
else:
    print("Paired t-test: No significant difference")

    Record_ID Machine_Type Operating_Mode Lubrication_Type  Temperature_C  \
0    REC_1000       Type_C            Eco          Mineral          85.80   
1    REC_1001       Type_A       Standard          Mineral          78.69   
2    REC_1002       Type_C            Eco          Mineral          84.75   
3    REC_1003       Type_C       Standard          Mineral          82.56   
4    REC_1004       Type_A       Standard        Synthetic          65.05   
..        ...          ...            ...              ...            ...   
195  REC_1195       Type_B       Standard        Synthetic          92.76   
196  REC_1196       Type_C       Standard          Mineral          75.93   
197  REC_1197       Type_C      High_Load          Mineral          64.66   
198  REC_1198       Type_B      High_Load        Synthetic          93.28   
199  REC_1199       Type_A      High_Load          Mineral          81.47   

     Vibration_mm_s  Pressure_kPa     RPM  Operating_Hours  \
0            

In [8]:
df.isnull().sum()

Machine_Type                   5
Operating_Mode                 0
Lubrication_Type               0
Temperature_C                  5
Vibration_mm_s                 0
Pressure_kPa                   5
RPM                            0
Operating_Hours                0
Remaining_Useful_Life_Hours    0
dtype: int64

In [9]:
# Group model validation fold scores
model_scores = [scores1, scores2]

# Compute mean RMSE for each model
model_means = [np.mean(scores) for scores in model_scores]

# Use np.argmin because lower RMSE indicates better performance
best_index = np.argmin(model_means)

print(f"Best model: Model {best_index + 1} with mean score = {model_means[best_index]:.4f}")

Best model: Model 2 with mean score = 740.6719


In [11]:
print(hello)

NameError: name 'hello' is not defined